In [32]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../market_data.db")

In [33]:
df = pd.read_sql("SELECT * FROM symbols", conn)
df

,ticker,sector,sub_sector
0,NVDA,Technology,Semiconductors
1,AMD,Technology,Semiconductors
2,MU,Technology,Semiconductors
3,SNOW,Technology,Software
4,CRM,Technology,Software
5,DDOG,Technology,Software
6,NVS,Healthcare,Drug Manufacturers
7,LLY,Healthcare,Drug Manufacturers
8,BIIB,Healthcare,Drug Manufacturers
9,UNH,Healthcare,Healthcare Plans


In [34]:
df = pd.read_sql("SELECT * FROM prices LIMIT 10", conn)
df

,ticker,date,open,high,low,close,volume
0,NVDA,2019-01-02,3.234882,3.429014,3.220272,3.373052,508752000
1,NVDA,2019-01-03,3.312882,3.346805,3.161835,3.169263,705552000
2,NVDA,2019-01-04,3.242310,3.410442,3.211605,3.372309,585620000
3,NVDA,2019-01-07,3.429509,3.587737,3.378252,3.550842,709160000
4,NVDA,2019-01-08,3.632308,3.634537,3.389890,3.462442,786016000
5,NVDA,2019-01-09,3.513700,3.577833,3.463186,3.530538,617260000
6,NVDA,2019-01-10,3.511223,3.604822,3.450804,3.596156,523156000
7,NVDA,2019-01-11,3.573870,3.708080,3.546137,3.685299,874764000
8,NVDA,2019-01-14,3.633051,3.750422,3.609528,3.725165,730168000
9,NVDA,2019-01-15,3.757850,3.797222,3.692727,3.711051,617012000


In [35]:
df = pd.read_sql("SELECT * FROM prices WHERE ticker = 'NVDA' ORDER BY date DESC LIMIT 10", conn)
df

,ticker,date,open,high,low,close,volume
0,NVDA,2026-09-10,220.485001,220.990005,217.199997,218.360001,100845811
1,NVDA,2026-09-09,225.279999,226.179993,223.460007,223.669998,82955500
2,NVDA,2026-09-08,233.110001,233.710007,224.850006,225.729996,122965600
3,NVDA,2026-09-04,231.089996,234.759995,229.630005,230.360001,135352400
4,NVDA,2026-09-03,226.020004,230.399994,224.750000,228.449997,134681600
5,NVDA,2026-09-02,218.789993,227.949997,218.479996,224.410004,157104700
6,NVDA,2026-09-01,216.750000,220.410004,215.100006,217.440002,109756200
7,NVDA,2026-08-31,218.869995,221.300003,216.210007,220.779999,124702700
8,NVDA,2026-08-28,227.360001,229.259995,216.809998,217.550003,195116400
9,NVDA,2026-08-27,222.860001,230.470001,220.899994,227.979996,298909800


*Aggregate functions*


In [ ]:
# min, max function
df = pd.read_sql("""
    SELECT ticker, date, MIN(close), MAX(close)
    FROM prices
    GROUP BY ticker""",conn)

df

,ticker,date,MIN(close),MAX(close)
0,ALHC,2021-06-21,4.470000,27.270000
1,AMD,2026-06-30,17.049999,580.909973
2,AXP,2025-12-11,63.396408,381.781555
3,BIIB,2021-06-10,113.379997,414.709991
4,BX,2024-11-22,22.046978,187.206390
5,CRM,2024-12-04,122.186119,363.219238
6,DDOG,2026-08-04,28.040001,288.149994
7,JPM,2026-08-12,66.463905,365.179993
8,KKR,2025-01-31,17.167788,165.185211
9,LLY,2026-08-19,97.758133,1280.339966


In [38]:
df = pd.read_sql("""
    SELECT COUNT(DISTINCT ticker)
    FROM prices
""", conn)
df

,COUNT(DISTINCT ticker)
0,18


In [48]:
#Sector level aggregation

query = """
    SELECT 
    symbols.sector,
    symbols.sub_sector,
    COUNT(DISTINCT prices.ticker) AS num_stocks,
    AVG(prices.close) AS avg_close,
    AVG(prices.volume) AS avg_volume
    FROM prices
    JOIN symbols ON prices.ticker = symbols.ticker
    GROUP BY symbols.sub_sector
"""

df = pd.read_sql(query, conn)
df

,sector,sub_sector,num_stocks,avg_close,avg_volume
0,Financial Services,Asset Management,2,80.510432,4.121556e+06
1,Financial Services,Banks,2,106.452729,1.763112e+07
2,Financial Services,Credit Services,2,212.422030,5.511323e+06
3,Healthcare,Drug Manufacturers,3,262.932295,2.405420e+06
4,Healthcare,Healthcare Plans,3,235.146987,2.544895e+06
5,Technology,Semiconductors,3,106.069093,1.585489e+08
6,Technology,Software,3,176.537511,5.892999e+06


In [43]:
#Daily return by stock

query = """
    SELECT 
    ticker, 
    date, 
    close, 
    LAG(close) OVER(PARTITION BY ticker) AS previous_close, 
    (close-LAG(close) OVER(PARTITION BY ticker))/LAG(close) OVER(PARTITION BY ticker) AS daily_return
    FROM prices
    LIMIT 10
"""

df = pd.read_sql(query, conn)
df

,ticker,date,close,previous_close,daily_return
0,ALHC,2021-03-26,17.309999,NaN,NaN
1,ALHC,2021-03-29,19.000000,17.309999,0.097631
2,ALHC,2021-03-30,19.940001,19.000000,0.049474
3,ALHC,2021-03-31,21.930000,19.940001,0.099799
4,ALHC,2021-04-01,23.200001,21.930000,0.057912
5,ALHC,2021-04-05,23.520000,23.200001,0.013793
6,ALHC,2021-04-06,23.590000,23.520000,0.002976
7,ALHC,2021-04-07,21.770000,23.590000,-0.077151
8,ALHC,2021-04-08,22.770000,21.770000,0.045935
9,ALHC,2021-04-09,23.830000,22.770000,0.046552
